In [156]:
import os
import pygmt
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
import numpy as np
import pandas as pd 
import glob 

%load_ext autoreload 
%autoreload 2
%matplotlib inline
import utils
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [157]:
cell_dict = {
    0.01: (20, 4096),
    0.008: (20, 4096),
    0.0063: (20, 4096),
    0.005: (10, 8192),
    0.0039: (10, 8192),
    0.003: (5, 16384),
    0.002: (5, 16384),
    0.0014: (2.5, 32768),
    0.001: (2.5, 32768) 
} # DX and N2 info for each model from Motorcycle shell scripts 

In [ ]:
#### this is the only cell that needs changing before running 
main_dir = "model_outputs/two_faults/inplane/"
files_interest = ["Dc_0063/output_Dc0063_120km/"] # can take in list
loading_type = "In plane"
dc = 0.0063 # Dc in RSF 
D = 120 # distance between the two parallel faults in kilometers
spin_up = 200  # n of years to omit in analysis - approx time it takes for steady state to be reached ~200 for anti and ~300 for in plane

In [159]:
DX, N2 = cell_dict[dc]
DX = [DX]
N2 = [N2]
cols = ['Index', 'Displacement', 'Col3', 'Col4', 'Col5', 'col', 'cplb','Slip Rate', 'Col7', 'Col8', 'Col9', 'Col10','Col11'] 
# time step conversion
time_step_rate = 20 # --export-netcdf-rate 20 \ ----- I am using the same in all models so this one shouldn't change..
grid_step_rate_horizontal = 4

feature = 'log10v' # tau, slip, log10v
fault1_path = main_dir + "fault-01-" + feature + ".grd"

In [160]:
time_step_rate = 20
grid_step_rate_horizontal = 4 

for i, (file, DXi, N2i) in enumerate(zip(files_interest, DX, N2)):
    file = file
    
    # load time file for times
    file_pattern = main_dir + file + 'patch-01-*.dat'
    matching_files = glob.glob(file_pattern)
    fault_data = pd.read_csv(matching_files[0], sep='\s+', header=None, names=cols)

    # load grid for moment
    feature = 'log10v' # tau, slip, log10v
    fault1_path = main_dir + file + "fault-01-" + feature + ".grd"
    fault2_path = main_dir + file + "fault-02-" + feature + ".grd"
    grid_fault1 = pygmt.load_dataarray(fault1_path, engine='netcdf4')
    grid_fault2 = pygmt.load_dataarray(fault2_path, engine='netcdf4')
    delta_grid = time_step_rate   
    time_grid = fault_data['Index'].iloc[::int(delta_grid)].values
    
    # downsample, otherwise gets massive for the big simulations
    if dc>0.005:
        grid_fault1 = grid_fault1.coarsen(x=2, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=2, boundary='trim').mean()
    elif 0.0014 <= dc <= 0.005:
        grid_fault1 = grid_fault1.coarsen(x=3, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=3, boundary='trim').mean()
    else:
        grid_fault1 = grid_fault1.coarsen(x=4, boundary='trim').mean()
        grid_fault2 = grid_fault2.coarsen(x=4, boundary='trim').mean()

    # make catalog
    grid1_masked = np.ma.masked_where(grid_fault1 < -3.5, grid_fault1) # subset areas where velocity>seismic slip to isolate events
    grid2_masked = np.ma.masked_where(grid_fault2 < -3.5, grid_fault2) # subset areas where velocity>seismic slip to isolate events
    grid1_masked = grid1_masked.filled(np.nan) # for viz in seaborn fill unmasked areas with nan
    grid2_masked = grid2_masked.filled(np.nan) # for viz in seaborn fill unmasked areas with nan
    timestep_event_fault1, rupture_length_pixels_fault1, group_id_fault1, total_pixels_fault1_xdir,pixel_size = utils.find_events(grid1_masked,DXi,N2i,"No")
    timestep_event_fault2, rupture_length_pixels_fault2, group_id_fault2, total_pixels_fault2_xdir,pixel_size = utils.find_events(grid2_masked,DXi,N2i,"No")
    time_f1 = time_grid[timestep_event_fault1]
    time_f2 = time_grid[timestep_event_fault2]
    rupture_length_fault1 = np.array(rupture_length_pixels_fault1) 
    rupture_length_fault2 = np.array(rupture_length_pixels_fault2) 

    # remove spin-up period 
    time_cutoff = spin_up * 365 * 24 * 60 * 60
    cutoff_index = np.where(time_grid >= time_cutoff)[0][0]
    time_grid = time_grid[cutoff_index:]
    time_f1 = time_f1[time_f1 >= time_cutoff]
    time_f2 = time_f2[time_f2 >= time_cutoff]
    rupture_length_fault1 = rupture_length_fault1[len(rupture_length_fault1) - len(time_f1):]
    rupture_length_fault2 = rupture_length_fault2[len(rupture_length_fault2) - len(time_f2):]

    # remove ruptures that are 4 cells only (1 cell in here because of MTC downsampling) -- artifacts from masking
    ok_size_rupture_idx1 = np.where(rupture_length_fault1 > 3)[0]
    ok_size_rupture_idx2 = np.where(rupture_length_fault2 > 3)[0]
    time_f1 = time_f1[ok_size_rupture_idx1]
    rupture_length_fault1 = rupture_length_fault1[ok_size_rupture_idx1]
    time_f2 = time_f2[ok_size_rupture_idx2]
    rupture_length_fault2 = rupture_length_fault2[ok_size_rupture_idx2]
    
    # remove partial ruptures (ruptures with length < 4.5 km)
    time_f1 = time_f1[rupture_length_fault1 * pixel_size > 4500]
    time_f2 = time_f2[rupture_length_fault2 * pixel_size > 4500]
    print(time_f1/(365 * 24 * 60 * 60) ,time_f2/(365 * 24 * 60 * 60))

    # measure interevent times
    inter_event_times_f1 = utils.measure_intevent_time_f(time_f1) # in seconds
    inter_event_times_years_f1 = inter_event_times_f1 / (365 * 24 * 60 * 60) # in years
    inter_event_times_f2 = utils.measure_intevent_time_f(time_f2) # in seconds
    inter_event_times_years_f2 = inter_event_times_f2 / (365 * 24 * 60 * 60) # in years

    print('Inter-event times fault 1 (years)', inter_event_times_years_f1, 'Inter-event times fault 2 (years)', inter_event_times_years_f2)
    
    times_file =  "code_output_data/interevent_times_two_faults_full_ruptures.csv"
    if not os.path.isfile(times_file):
        cols = ["Loading", "Dc", "D", "Inter-event times fault 1 (seconds)", "Inter-event times fault 1 (years)", "Inter-event times fault 2 (seconds)", "Inter-event times fault 2 (years)"]
        times_database = pd.DataFrame(columns=cols)
        times_database.to_csv(times_file, index=False)
    else:
        times_database = pd.read_csv(times_file)
    existing_combination = times_database[(times_database['Dc'] == dc) & (times_database['D'] == D) & (times_database['Loading'] == loading_type)]
    if not existing_combination.empty:
        times_database = times_database[~((times_database['Dc'] == dc) & (times_database['D'] == D) & (times_database['Loading'] == loading_type))]
        print(f"Removed existing rows with Dc={dc} and D={D}.")

    times_sim_i = pd.DataFrame({
        'Loading': [loading_type],
        'Dc': [dc],
        'D':D,
        'Inter-event times fault 1 (seconds)': [", ".join(map(str, inter_event_times_f1))],
        'Inter-event times fault 1 (years)': [", ".join(map(str, inter_event_times_years_f1))],
        'Inter-event times fault 2 (seconds)': [", ".join(map(str, inter_event_times_f2))],
        'Inter-event times fault 2 (years)': [", ".join(map(str, inter_event_times_years_f2))]
    })
    
new_rows = pd.concat([times_sim_i], ignore_index=True)
catalog_df = pd.concat([times_database, new_rows], ignore_index=True)
catalog_df.to_csv(times_file, index=False)
print("New row added to interevent time file.csv")

[ 218.3724774   256.0468551   293.7453127   331.45457638  369.16873819
  406.88521082  444.60274613  482.32082963  520.03923607  557.7578281
  595.47653309  633.19545305  670.91445317  708.63357304  746.35281533
  784.07217038  821.79165021  859.51124376  897.23093237  934.95080582
  972.67076575 1010.39080592 1048.1110484  1085.83137466] [ 227.77572231  265.47295405  303.19050471  340.9172163   378.64804769
  416.38076754  454.11437215  491.84837734  529.58264263  567.31707927
  605.05158188  642.78615221  680.52079595  718.25554012  755.99034224
  793.72516895  831.46011402  869.19509244  906.93012753  944.66523511
  982.40041563 1020.13565322 1057.87094772 1095.6062875 ]
Inter-event times fault 1 (years) [37.67437771 37.69845759 37.70926368 37.71416181 37.71647263 37.71753531
 37.7180835  37.71840644 37.71859203 37.71870499 37.71891995 37.71900013
 37.71911987 37.71924229 37.71935505 37.71947983 37.71959355 37.71968861
 37.71987345 37.71995992 37.72004018 37.72024248 37.72032626] In